# Cross-Cancer-Type Comparison: RCC Selection Findings vs. CRC Gene Frequencies (Puccini et al. 2021)

**What this notebook is — and isn't.**

This is **not** a replication of our TRACERx Renal classifier in a new cancer type. Puccini et al.'s dataset (11,871 largely *unmatched* CRC samples, Caris Life Sciences) answers a structurally different question: *how often is gene X mutated in primary vs. lymph node vs. distant metastasis samples, across a population?* Our RCC analysis answers: *given a mutation that was subclonal in a patient's own primary tumor, does it become clonally dominant in that same patient's metastasis?* These are related but distinct questions, and the datasets cannot be merged into one model.

**What we can honestly do:** check whether genes that stood out in our RCC selection analysis show a *directionally consistent* pattern in Puccini's independent CRC population — as a qualitative, hypothesis-generating cross-cancer comparison, reported systematically (all overlapping genes, not just the ones that look good) to avoid cherry-picking.

In [2]:
import pandas as pd

# RCC side: our own selection-rate results (point mutations, SNV_INDEL_DNV sheet)
rcc = pd.read_csv("binary_selection_dataset.csv")
rcc_snv = rcc[rcc["event_category"] == "SNV_INDEL_DNV"]

# CRC side: Puccini et al.'s full 388-row NGS gene panel (Supplemental Data, figshare 14854383)
crc = pd.read_excel("Supplemental_Data.xlsx", sheet_name="NGS Data", header=0)
crc.columns = ["gene", "lymph_pos", "lymph_neg", "lymph_total", "lymph_pct",
               "blank1", "other_pos", "other_neg", "other_total", "other_pct",
               "blank2", "primary_pos", "primary_neg", "primary_total", "primary_pct",
               "blank3", "p_lymph_vs_primary", "p_lymph_vs_other"] + [f"extra{i}" for i in range(len(crc.columns)-18)]
crc = crc.dropna(subset=["gene"])
crc["gene_clean"] = crc["gene"].str.replace("NGS-", "", regex=False)
print(f"RCC point-mutation events: {len(rcc_snv)}   CRC gene panel size: {len(crc)}")

RCC point-mutation events: 230   CRC gene panel size: 144


## Systematic overlap: every gene present in both the TRACERx Renal driver panel and Puccini's CRC panel

No selection of "interesting" genes here — this is every gene that happens to appear in both datasets' panels.

In [3]:
tracerx_genes = ["TP53", "SETD2", "PBRM1", "VHL", "PIK3CA", "MTOR", "PTEN", "KDM5C", "BAP1",
                 "TSC1", "TSC2", "CDKN2A", "ATM", "ATRX", "BRCA1"]

rows = []
for g in tracerx_genes:
    rcc_sub = rcc_snv[rcc_snv["gene_or_locus"] == g]
    crc_sub = crc[crc["gene_clean"] == g]
    if len(crc_sub) == 0:
        continue
    crc_row = crc_sub.iloc[0]
    rows.append({
        "gene": g,
        "RCC_n": len(rcc_sub),
        "RCC_selection_rate": rcc_sub["label"].mean() if len(rcc_sub) > 0 else None,
        "CRC_lymph_pct": crc_row["lymph_pct"],
        "CRC_distant_pct": crc_row["other_pct"],
        "CRC_primary_pct": crc_row["primary_pct"],
        "CRC_p_lymph_vs_primary": crc_row["p_lymph_vs_primary"],
        "CRC_p_lymph_vs_distant": crc_row["p_lymph_vs_other"],
    })

comparison = pd.DataFrame(rows).sort_values("RCC_selection_rate", ascending=False)
comparison

,gene,RCC_n,RCC_selection_rate,CRC_lymph_pct,CRC_distant_pct,CRC_primary_pct,CRC_p_lymph_vs_primary,CRC_p_lymph_vs_distant
5,MTOR,9,0.555556,0.000000,0.007697,0.011031,0.103,0.173
8,BAP1,10,0.500000,0.004132,0.001512,0.004666,0.906,0.338
2,PBRM1,19,0.421053,0.008403,0.006751,0.012700,0.562,0.766
7,KDM5C,10,0.300000,0.000000,0.006547,0.009110,0.278,0.359
9,TSC1,5,0.200000,0.012448,0.005478,0.005774,0.201,0.175
0,TP53,7,0.142857,0.723164,0.665473,0.693608,0.242,0.026
1,SETD2,28,0.071429,0.008511,0.007403,0.020278,0.207,0.849
6,PTEN,5,0.000000,0.041543,0.030621,0.045188,0.754,0.266
4,PIK3CA,7,0.000000,0.115493,0.152002,0.179173,0.002,0.062
10,TSC2,4,0.000000,0.004132,0.004561,0.006595,0.643,0.924


## Reading this table honestly

**Genes with a plausible directional parallel:**
- **SETD2** — the clearest case. In our RCC selection analysis, SETD2 point mutations have the **lowest selection rate among reasonably-sized gene groups (7.1%, n=28)** — i.e. SETD2-mutant subclones rarely become clonally dominant in the metastasis. In Puccini's independent CRC population, SETD2 is markedly **depleted in both lymph node (0.85%) and distant metastases (0.74%) relative to primary tumors (2.03%)**. Two independent cancer types, two independent cohorts, two structurally different analyses — both point toward SETD2-mutant clones being relatively poor metastatic seeders.
- **PIK3CA** — RCC selection rate is 0% (n=7, small), and in CRC, PIK3CA is significantly depleted in lymph node vs. primary (11.5% vs 17.9%, p=0.002). Directionally consistent, though the RCC sample size here is too small to lean on.
- **TP53/17p** — not directly comparable at the point-mutation level in our RCC panel (n=7 only), but our **arm-level 17p SCNA finding was the most robust FDR-significant locus in the entire RCC locus-enrichment screen** (see `Locus_Enrichment_Validation.ipynb`), and TP53 is the single most differentially represented gene in Puccini's CRC comparison (highest prevalence in LN at 72.3%, significantly different from distant, p=0.026). The TP53/17p axis being prominent in both cancer types — via different exact mechanisms (SCNA selection in RCC, LN-specific mutation prevalence in CRC) — is a reasonable point of convergence to report.

**Genes with no clear parallel (reported for completeness, not omitted):**
- BAP1, MTOR, KDM5C show *high* RCC selection rates (42-56%) but no strong or consistent CRC lymph/distant differential
- ATM and BRCA1 show significant CRC lymph-vs-distant differences (p=0.005 each) that don't have a corresponding strong RCC selection signal in our data (small n for ATM in RCC)
- VHL, TSC1, TSC2, CDKN2A, PTEN, PBRM1, ATRX show no compelling cross-cancer parallel either direction

This is exactly what an honest hypothesis-generating cross-check should look like: a couple of genuinely interesting convergent signals (SETD2 above all), a plausible secondary one (TP53/17p), and several genes that simply don't line up — reported rather than hidden.

## Conclusion for the paper

This cross-cancer-type comparison **cannot** serve as a formal replication of our RCC classifier — the two studies ask different questions on structurally different data (matched multi-region subclonal transitions vs. population-level site prevalence in largely unmatched samples). What it *can* responsibly contribute to the paper is a **discussion-section observation**: the finding that SETD2-mutant clones are relatively metastasis-incompetent recurs across two independent cancer types and two independent analytical frameworks, which is a modest but genuine piece of convergent evidence worth reporting — alongside the honest caveat that most other genes tested show no cross-cancer parallel, and that a true generalization test would require a CRC dataset with the same matched multi-region clonal/subclonal schema as TRACERx (e.g. Wei et al. 2017, Hu et al. 2019), which remains a natural direction for future work rather than something this paper can claim to have done.